# Environment Setup

Examples below are set to run in the package's `test` folder:

In [1]:
import inu.ds
from pathlib import Path
labels_str = lambda p: ', '.join(map(lambda kv: "{}: {}".format(*kv), p.items()))
test_path = Path(inu.ds.__file__).parent / 'tests'
%cd {test_path}

/home/ilyap/code/algodev/inu/ds/tests


# Dataset package `inu.ds`

## Introduction

`inu.ds` package contains set of tools to work with file-system based datsets.  

It supports several layers of abstruction, from simple paths parsing functions,  
to files browsing, to high level API for labeled data access and processing,  
in the modules:

    scan        - scanning the files structure.
    collect     - collecting data elements into tabular labeled form.
    transform   - transforming the data elements along typical pipelines.
    label - ensures pre-defined label requirement are met and add labeling fuctionality.
    scheme - scheme specific object that allows customized labeled data process and iteration
    catalog - aggregated structure to hold datasets
    torch - supplies an interface between Inuitive known dataset handling such as DataCollection to torch official API.
    

## Describe Dataset Files Structure
**Given**: 
- Dataset files organized in sub-folders according to certain _template_
- Path to every data file may include meta infomation _labeling_ the corresponding data elements
- Collection of those _labels_ both _describe_ and _identify_ all the data elements

**Objective**:
 - Formally describe the file-system structure of the dataset including the _labeling_ system.
   
### Example   
Consider file structure under folder `../inu/ds/tests/data`:

In [2]:
!tree data

data
├── level_one_1
│   ├── file_sceneA_id0.txt
│   └── file_sceneA_id1.txt
├── level_one_2
│   ├── file_sceneA_id0.txt
│   └── file_sceneA_id1.txt
├── level_two_1
│   ├── file_sceneB_id0.txt
│   └── file_sceneB_id1.txt
├── level_two_2
│   ├── file_sceneB_id0.txt
│   └── file_sceneB_id1.txt
└── scheme.yml

4 directories, 9 files


This files naming structure can be formalized by a single template.

### Parsing Templates
The files names contain labels according to certain _parsing template_ .  
Note that arbitrary _labels names_ could be introduced,  
and the corresponding _labels values_ are specific for every particular file.

Parsing template can be provided in either `format` or `regex` form, or even in a mixed form.
Several functions are defined to translate between them if needed:

In [3]:
from inu.ds.scheme import format_to_regex, regex_to_format, unmix_regex_format

The `format` form defines lables as python's `str.format()` function inside curled braces:

In [4]:
template_f = r'level_{count}_{level}/file_scene{scene}_id{view}'

Same template can be define in `regex` form.   
It allows to use all the expressivity of the 
regular expression syntax on account of higher complexity.  
In simple case `format` form is equivalent to the `regex` form 
with `{label}` replaced by `(?P<label>\w+)`:

In [5]:
print(format_to_regex(template_f))  # to equivalent regex form

level_(?P<count>\w+)_(?P<level>\w+)/file_scene(?P<scene>\w+)_id(?P<view>\w+)


However, more particular rules may be given if required:

In [6]:
template_r = r'level_(?P<count>one|two)_(?P<level>\d+)/file_scene(?P<scene>[A-Z])_id(?P<view>\d+)'
print(regex_to_format(template_r))  # back to the fomat form

level_{count}_{level}/file_scene{scene}_id{view}


For simplicity a mixed expressions are supported, when simple lables introduces by the `form` pattern:

In [7]:
template_m = r'level_{count}_(?P<level>\d+)/file_scene{scene}_id(?P<view>\d+)'

In [8]:
[print(f) for f in unmix_regex_format(template_m)];  # to pure regex and format forms

level_(?P<count>\w+)_(?P<level>\d+)/file_scene(?P<scene>\w+)_id(?P<view>\d+)
level_{count}_{level}/file_scene{scene}_id{view}


## Scan & Parse Paths
Once the expected file system structure is defined it can be scanned according to the parsing pattern using functions from `inu.ds.scan` .       
Those functions are generators of paths matching the specified pattern. They support many different use cases through their arguments.

In [9]:
from inu.ds.scan import match_paths, match_sub_files

For example to generate tuples (path, labels) matching regex `template_r` in the current folder:

In [10]:
for p in match_paths('data', template_r):
    print(p[0], labels_str(p[1]))

data/level_one_1/file_sceneA_id1.txt count: one, level: 1, scene: A, view: 1
data/level_one_1/file_sceneA_id0.txt count: one, level: 1, scene: A, view: 0
data/level_two_2/file_sceneB_id0.txt count: two, level: 2, scene: B, view: 0
data/level_two_2/file_sceneB_id1.txt count: two, level: 2, scene: B, view: 1
data/level_two_1/file_sceneB_id0.txt count: two, level: 1, scene: B, view: 0
data/level_two_1/file_sceneB_id1.txt count: two, level: 1, scene: B, view: 1
data/level_one_2/file_sceneA_id1.txt count: one, level: 2, scene: A, view: 1
data/level_one_2/file_sceneA_id0.txt count: one, level: 2, scene: A, view: 0


If only files in specific folder are needed `match_sub_files()` provides shorter syntax:

In [11]:
folder = './data/level_one_2'
!ls -m {folder}
[*match_sub_files(folder, format_to_regex('file_scene{scene}_id1.{ext}'), ret='name')]

file_sceneA_id0.txt, file_sceneA_id1.txt


[('file_sceneA_id1.txt', {'scene': 'A', 'ext': 'txt'})]

### Efficient Functions to `walk` the Files Structure
The `match_*`  functions exuastively go over __all__ the folders and files in certain directory and filter those matching conditions given by a regular expression.  
More effective way for large datasets would be walk the file system guided by the the required conditions and skip not relevant paths altogether.  

- `walk_path` supports different types of conditions, _excluding_ regular expressions for paths 
- `walk_scheme` supports regular expressions and parsing, thus can also extract lables for every path generated



In [12]:
from inu.ds.scan import walk_paths, walk_scheme
[*walk_paths('data', dirs=True, depth=2, guide=lambda p: 'one' in str(p))]

['data/level_one_1', 'data/level_one_2']

In [13]:
rex = format_to_regex('level_{count}_{level}/file_scene{scene}_id{view}.txt')
for labels, path in walk_scheme('./data', rex):
    print(f'{path} => {labels_str(labels)}')

data/level_one_1/file_sceneA_id1.txt => scene: A, view: 1, count: one, level: 1
data/level_one_1/file_sceneA_id0.txt => scene: A, view: 0, count: one, level: 1
data/level_two_2/file_sceneB_id0.txt => scene: B, view: 0, count: two, level: 2
data/level_two_2/file_sceneB_id1.txt => scene: B, view: 1, count: two, level: 2
data/level_two_1/file_sceneB_id0.txt => scene: B, view: 0, count: two, level: 1
data/level_two_1/file_sceneB_id1.txt => scene: B, view: 1, count: two, level: 1
data/level_one_2/file_sceneA_id1.txt => scene: A, view: 1, count: one, level: 2
data/level_one_2/file_sceneA_id0.txt => scene: A, view: 0, count: one, level: 2


In [14]:
from inu.ds import PathScheme, DataCollection, CacheMode
from param import config

## Universal Data Model
This model considers any data set as a collection of labeled elements:
- Information of the element is organized as a set of labels:  
   `{category: value}` - where `category` is a _string_ and the `value` may be of any type   
- Different elements may contain different kinds of labels - but enough to allow unique indexing
- There is _always_ 'path' label assigned to every data element with a full path to its source file. 
- Programmically that means data element is represented by a _dictionary_.

## Scheme of a Datset
File system based datasets may include implicit information not only in the names and structure of the paths, but also:
 - assumptions about meaning of lables (aliasing)
 - constrains on allowed combination of labels

`PathScheme` allows to make this information explicit and describe dataset fully and unambiguously.  
- It provides a _universal_ dataset-independent way to iterate over its elements.  
- Universal access to the data elements is made possible by the _Universal Data Model_.

Thus `PathScheme` main role to provide iterator over `data elements` interface for datsets:  

    Files Structure -> PathScheme -> Iterator -> DataElements

It can be considered as `walk_scheme(template)` elevated to class to support additional functionality:

In [15]:
scheme = PathScheme(rex) # scheme defined as an abstract structure not connected to particular path
for p in scheme.iter(root='data', cache=CacheMode.PASS):  # path is required to create an iterator, though
    print(labels_str(p))

count: one, level: 1, scene: A, view: 1, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id1.txt
count: one, level: 1, scene: A, view: 0, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
count: two, level: 2, scene: B, view: 0, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
count: two, level: 2, scene: B, view: 1, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
count: two, level: 1, scene: B, view: 0, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
count: two, level: 1, scene: B, view: 1, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
count: one, level: 2, scene: A, view: 1, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id1.txt
count: one, level: 2, scene: A, view: 0, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt


Advanced capabilities of `PathScheme` include conditions and aliases introducing particular semantic meaning to the dataset:

In [16]:
# scheme = PathScheme.from_yaml(
# r"""
#     pattern: 'data/level_{count}_{level}/file_scene{scene}_id{side}.txt'
#     mappings:
#         side: {'0':'L', '1':'R'}
#         level: {'1':'easy', '2':'hard'}
# """
# , filters="side=='L' or count=='two'")

scheme = PathScheme(rex, 
                    mappings=dict(
                        view={'0':'L', '1':'R'},
                        level={'1':'easy', '2':'hard'}),
                    filters="view=='L' or count=='two'")

                    #cond="side=='L' or count=='two'")
for p in scheme.iter(root='data'):  # by default parses the current folder
    print(labels_str(p))

count: one, level: easy, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
count: two, level: hard, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
count: two, level: hard, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
count: two, level: easy, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
count: two, level: easy, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
count: one, level: hard, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt


### Schemes in YAML
Sometimes its preferable to handle schemes not inside the code but as a declarative structures.  
Then `PathScheme` can be created from `yaml` file or string:

In [17]:
%%file scheme.yml
    pattern: 'data/level_{count}_{level}/file_scene{scene}_id{view}.txt'
    mappings:
        view: {'0':'L', '1':'R'}
        level: {'1':'easy', '2':'hard'}

Writing scheme.yml


In [18]:
scheme = PathScheme.from_yaml('scheme.yml')
for p in scheme.iter(root='.'):
    print(labels_str(p))

count: one, level: easy, scene: A, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id1.txt
count: one, level: easy, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
count: two, level: hard, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
count: two, level: hard, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
count: two, level: easy, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
count: two, level: easy, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
count: one, level: hard, scene: A, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id1.txt
count: one, level: hard, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt


In [19]:
scheme = PathScheme(**config(r"""
        pattern: 'data/level_{count}_{level}/file_scene{scene}_id{view}.txt'
        mappings:
            view: {'0':'L', '1':'R'}
            level: {'1':'easy', '2':'hard'}
        filters: "view=='L' or count=='two'"
"""))
for p in scheme.iter(root='.'):
    print(labels_str(p))

count: one, level: easy, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
count: two, level: hard, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
count: two, level: hard, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
count: two, level: easy, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
count: two, level: easy, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
count: one, level: hard, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt


#### Automatically load YAML Scheme from Dataset Folder
If the _root_ folder of a _dataset_ contains `scheme.yml` file or
`info.yml` file with `scheme` node, then scheme can be automatically loaded given the path:

In [20]:
scheme = PathScheme.from_yaml(root='.')
for p in scheme.iter(root='.'):
    print(labels_str(p))

count: one, level: easy, scene: A, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id1.txt
count: one, level: easy, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
count: two, level: hard, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
count: two, level: hard, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
count: two, level: easy, scene: B, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
count: two, level: easy, scene: B, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
count: one, level: hard, scene: A, view: R, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id1.txt
count: one, level: hard, scene: A, view: L, path: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt


## Data Collection

`DataCollection` is a _container_ built around pandas `DataFrame` to hold labels of _all_ the dataset's elements.  
It uses `PathScheme`s to access one or more dataset and load them into a single table (_DataFrame_).  
That allows to support advanced manipulations on the labels, like queries and data transformation.

Most of `DataCollection` capabilities are built around its main attribute `.db`

In [21]:
scheme = PathScheme(rex, root='data', 
                    mappings=dict(
                        view={'0':'L', '1':'R'},
                        level={'1':'easy', '2':'hard'}))

In [22]:
dc = DataCollection(scheme)
dc.db

path  \
count level scene view                                                      
one   easy  A     R     🖿…u/ds/tests/data/level_one_1/file_sceneA_id1.txt   
                  L     🖿…u/ds/tests/data/level_one_1/file_sceneA_id0.txt   
two   hard  B     L     🖿…u/ds/tests/data/level_two_2/file_sceneB_id0.txt   
                  R     🖿…u/ds/tests/data/level_two_2/file_sceneB_id1.txt   
      easy  B     L     🖿…u/ds/tests/data/level_two_1/file_sceneB_id0.txt   
                  R     🖿…u/ds/tests/data/level_two_1/file_sceneB_id1.txt   
one   hard  A     R     🖿…u/ds/tests/data/level_one_2/file_sceneA_id1.txt   
                  L     🖿…u/ds/tests/data/level_one_2/file_sceneA_id0.txt   

                       read_trans  
count level scene view             
one   easy  A     R       O|read|  
                  L       O|read|  
two   hard  B     L       O|read|  
                  R       O|read|  
      easy  B     L       O|read|  
                  R       O|read|  
one   hard  A     R       O|read|  
                  L       O|read|

### Data Collection Structure
 - All the data elements are merged into one (possibly _sparse_) 2D table with 
     - (_mult-level) row index_  - __0__-_axis_, and 
     - _named columns_ - __1__-_axis_,       
 - Every label can play a role of a data category (column) or index name (level)
 
By default all the labels categories except of the mandatory `path` are used as index levels.  
Thus, multi-level index is essentially a table by itself, as can be easily seen:

In [23]:
mi = dc.db.index
mi.to_frame(False) 

,count,level,scene,view
0,one,easy,A,R
1,one,easy,A,L
2,two,hard,B,L
3,two,hard,B,R
4,two,easy,B,L
5,two,easy,B,R
6,one,hard,A,R
7,one,hard,A,L


Particular categories can be explicetely set as data columns using `DataFrame` methods like `reindex` or during construction:

In [24]:
dc = DataCollection(scheme, data=['level','path'])
dc.db

level                                               path
count scene view                                                         
one   A     R     easy  🖿…u/ds/tests/data/level_one_1/file_sceneA_id1.txt
            L     easy  🖿…u/ds/tests/data/level_one_1/file_sceneA_id0.txt
two   B     L     hard  🖿…u/ds/tests/data/level_two_2/file_sceneB_id0.txt
            R     hard  🖿…u/ds/tests/data/level_two_2/file_sceneB_id1.txt
            L     easy  🖿…u/ds/tests/data/level_two_1/file_sceneB_id0.txt
            R     easy  🖿…u/ds/tests/data/level_two_1/file_sceneB_id1.txt
one   A     R     hard  🖿…u/ds/tests/data/level_one_2/file_sceneA_id1.txt
            L     hard  🖿…u/ds/tests/data/level_one_2/file_sceneA_id0.txt

### Iterations over Collection
There are several approaches to iterate over a data collection (`dc`):
1. Use direct acces to the `DataCollection.db` atribute and utilize all the relevent `DataFrame` methods
2. Use `DataCollection` as an iterator 
3. Use `DataCollection.iter(...)` function to create special iterators

#### `DataCollection` as Iterator
It's actually shortcuts `dc.db.iterrows()` and yields for every row a tuple (_index_, _series_).   
Series provides named access to the data categories and the index (._name_):

In [25]:
for idx, it in dc:    
    print(f'{idx} == {it.name}: level={it.level}, path={it.path}')

('one', 'A', 'R') == ('one', 'A', 'R'): level=easy, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id1.txt
('one', 'A', 'L') == ('one', 'A', 'L'): level=easy, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
('two', 'B', 'L') == ('two', 'B', 'L'): level=hard, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
('two', 'B', 'R') == ('two', 'B', 'R'): level=hard, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt
('two', 'B', 'L') == ('two', 'B', 'L'): level=easy, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
('two', 'B', 'R') == ('two', 'B', 'R'): level=easy, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
('one', 'A', 'R') == ('one', 'A', 'R'): level=hard, path=/home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id1.txt
('one', 'A', 'L') == ('one', 'A', 'L'): level=hard, path=/home/ilyap/code/al

#### Itrerate over Groups
Sometimes data elements must be processed not one by one, but in groups organized by their semantic meaning.
Groups are indexed (and identified) by set of categories 
To define a grouping iterator categories used as groups indices must be specified
In the example above the processing context couls be defined by all the data related to specific scene captured under certain conditions.

In [26]:
scheme = PathScheme.from_yaml(root='.')  # automatically load yaml-scheme from the dataset folder
dc = DataCollection(scheme, data='path')
print(f'{dc.categories} = {set(dc.db.index.names)} + {set(dc.db.columns)}')

{'path', 'scene', 'level', 'view', 'count'} = {'scene', 'level', 'count', 'view'} + {'path'}


In this example we group items by their (__scene__, __lid__) and use __view__ as index inside the groups:

In [27]:
for grp, gdata in dc.iter(group=('scene', 'level'), index='view', trans=False, data='path'):
    print('_'*50, f'\nGroup [{grp.scene}/{grp.level}]:')    
    print(gdata)

__________________________________________________ 
Group [A/easy]:
                                          path (Series)
view                                                   
L     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
R     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
__________________________________________________ 
Group [A/hard]:
                                          path (Series)
view                                                   
L     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
R     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
__________________________________________________ 
Group [B/easy]:
                                          path (Series)
view                                                   
L     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
R     /home/ilyap/code/algodev/inu/ds/tests/data/lev...
__________________________________________________ 
Group [B/hard]:
                                          path (Series)


Items inside the data-frames are accessible using the `DataFrame` indexing technics.  
For example column can be selected by `.column_name`, so it could be convoinient to transpose (turn) the data-frame:

In [28]:
for grp, paths in dc.iter(group=('scene', 'level'), index='view', trans=False, data='path'):
    print(f'{grp.scene}/{grp.level}: {paths.L}\n     <> {paths.R}')

A/easy: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id0.txt
     <> /home/ilyap/code/algodev/inu/ds/tests/data/level_one_1/file_sceneA_id1.txt
A/hard: /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id0.txt
     <> /home/ilyap/code/algodev/inu/ds/tests/data/level_one_2/file_sceneA_id1.txt
B/easy: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id0.txt
     <> /home/ilyap/code/algodev/inu/ds/tests/data/level_two_1/file_sceneB_id1.txt
B/hard: /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id0.txt
     <> /home/ilyap/code/algodev/inu/ds/tests/data/level_two_2/file_sceneB_id1.txt


# Caching

`PathScheme` and `DataCollection` supports cache storing and handling by default.

The current supported modes are:
 - `PASS` = to bypass caching mechanism
 - `LOAD` - use available cache files, raise if not found!
 - `SAVE` - saves FULL iterations results into the file
 - `KEEP` - same as `LOAD` if file is found otherwise `SAVE`


Notice the default mode is:
```python 
CacheMode.KEEP
```

The cache defaults can be changed by passing _cache_ arguemnt to:
`PathScheme`:
  
``` python 
PathScheme(cache=CacheMode.LOAD)

PathScheme.from_yaml(cache=CacheMode.PASS) 
``` 

as well as `DataCollection`:
```python 
DataCollection(cache=CacheMode.SAVE)
```

For further information check _inu/utils/cache.py_

# Deleting temporal resources

In [29]:
!rm scheme.yml
!rm -r .cache